# Cross-Chain Atomic Settlement — One Secret, Two Stablecoins

Notebook 8 separated TitusCoin's **on-chain token ledger** from Fictional Titus Bank's **off-chain reserve record**. Now TitusCoin (`TTC`) lives on Chain-A while fictional NovaCoin (`NVC`) lives on independent Chain-B.

They both target one fictional dollar. Swapping them is not a currency trade. It is a **domain trade**: different chains, different issuers, different guest lists for who may freeze or seize a balance. Alice on Chain-A wants NVC she can spend on Chain-B's rails; Bob wants TTC on Chain-A. Same face value, different rooms.

The new question is narrow: how can Titus Bank and Nova Bank exchange tokens so that both transfers complete—or both can unwind—without trusting one bank to pay first? A **hash time-locked contract (HTLC)** coordinates that outcome with one secret, one shared hash, and two deadlines. It is not one atomic transaction across chains. It is two escrows and a public confession.

In [2]:
import hashlib
import json
import random

from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Validator
from blockchain_lib.stablecoin import (
    Blockchain,
    FiatBackedIssuer,
    TokenLedger,
)


def sha256(data: str) -> str:
    """Return the SHA-256 hex digest used as an HTLC hash lock."""
    return hashlib.sha256(data.encode()).hexdigest()


## 1. Two chains, two issuers, four ledgers

Each token still has two books:

- Chain-A records `TTC` ownership and total supply; `titus_issuer` records fictional TTC reserves off-chain.
- Chain-B records `NVC` ownership and total supply; `nova_issuer` records fictional NVC reserves off-chain.

The chains share no validators or state. Each issuer first records 1,000 fictional USD of backing, then mints 1,000 tokens to its bank.

In [4]:
print("=== SETUP: two completely separate chains, one stablecoin each ===\n")

network_a = Network(["Node-A1", "Node-A2"], random.Random(1))
network_b = Network(["Node-B1", "Node-B2"], random.Random(2))

chain_a = Blockchain(
    "Chain-A",
    [Validator("Node-A1", 200), Validator("Node-A2", 200)],
)
chain_b = Blockchain(
    "Chain-B",
    [Validator("Node-B1", 300), Validator("Node-B2", 100)],
)

tituscoin = TokenLedger("TTC", chain_a, network_a)
novacoin = TokenLedger("NVC", chain_b, network_b)
titus_issuer = FiatBackedIssuer("Fictional Titus Bank", tituscoin)
nova_issuer = FiatBackedIssuer("Fictional Nova Bank", novacoin)

titus_issuer.deposit_and_mint("Titus Bank", 1_000)
nova_issuer.deposit_and_mint("Nova Bank", 1_000)

print(
    f"\n  Starting balances -- TTC: {tituscoin.balances} | "
    f"NVC: {novacoin.balances}\n"
)
print(
    f"  Fictional reserves -- TTC: ${titus_issuer.reserve_usd:,.0f} | "
    f"NVC: ${nova_issuer.reserve_usd:,.0f}\n"
)


=== SETUP: two completely separate chains, one stablecoin each ===


  Starting balances -- TTC: {'Titus Bank': 1000} | NVC: {'Nova Bank': 1000}

  Fictional reserves -- TTC: $1,000 | NVC: $1,000



## 2. The HTLC mechanism

Titus Bank chooses a secret but publishes only its hash. Both banks lock tokens against that **same hash**. Knowing the secret releases a lock; reaching a deadline without a claim permits a refund.

The crucial choreography is that claiming NVC writes the secret into Chain-B's public history. Nova Bank can read it there and use it to claim TTC on Chain-A. `HTLCManager` is the only new mechanism in this notebook — defined inline here; [`blockchain_lib/htlc.py`](../blockchain_lib/htlc.py) is the copy a later notebook would import.

Lock events still pass through notebook 5's waiting room: the ledger gossips a `Transaction` and `include`s it. Two chains means two mempools. Gossip on Chain-A does not fill an inbox on Chain-B.

In [6]:
class HTLCManager:
    """Manage hash-locked, time-limited escrows on one chain."""

    def __init__(self, blockchain: Blockchain, ledger: TokenLedger) -> None:
        self.blockchain = blockchain
        self.ledger = ledger
        self.locks: dict[str, dict] = {}

    def lock(
        self,
        sender: str,
        receiver: str,
        amount: float,
        hash_lock: str,
        timelock_height: int,
    ) -> bool:
        """Escrow funds behind a hash until they are claimed or expire."""
        if not self.ledger.lock_for_htlc(sender, amount, hash_lock):
            return False
        self.locks[hash_lock] = {
            "sender": sender,
            "receiver": receiver,
            "amount": amount,
            "timelock_height": timelock_height,
            "claimed": False,
        }
        return True

    def claim(self, hash_lock: str, secret: str) -> bool:
        """Reveal a matching secret and release the escrow to its receiver."""
        lock = self.locks.get(hash_lock)
        if lock is None or lock["claimed"] or sha256(secret) != hash_lock:
            print(f"  [{self.blockchain.name}] CLAIM FAILED.")
            return False
        lock["claimed"] = True
        record = json.dumps(
            {
                "event": "HTLC_CLAIM",
                "receiver": lock["receiver"],
                "amount": lock["amount"],
                "hash_lock": hash_lock,
                "revealed_secret": secret,
            },
            sort_keys=True,
        )
        self.ledger.record_payload(record)
        print(
            f"  [{self.blockchain.name}] CLAIM: "
            f"secret revealed on-chain = {secret!r}"
        )
        self.ledger.release_to(lock["receiver"], lock["amount"])
        return True

    def refund(self, hash_lock: str, current_height: int) -> bool:
        """Return unclaimed escrow after its supplied height expires."""
        lock = self.locks.get(hash_lock)
        if (
            lock is None
            or lock["claimed"]
            or current_height < lock["timelock_height"]
        ):
            return False
        record = json.dumps(
            {
                "event": "HTLC_REFUND",
                "sender": lock["sender"],
                "amount": lock["amount"],
            },
            sort_keys=True,
        )
        self.ledger.record_payload(record)
        self.ledger.refund_to(lock["sender"], lock["amount"])
        return True


## 3. Successful swap: lock the longer side first

> **Pause and predict:** Why does Chain-A use height 50 while the second lock on Chain-B uses the shorter height 30?

The second lock expires first. Even if Titus Bank reveals the secret near Chain-B's deadline, Nova Bank still has the extra window before height 50 to claim on Chain-A. Reversing that ordering could let the first lock expire before the final claim arrives.

In [8]:
htlc_a = HTLCManager(chain_a, tituscoin)
htlc_b = HTLCManager(chain_b, novacoin)

print("=== SCENARIO: Titus Bank swaps 1000 TTC for Nova Bank's 1000 NVC ===\n")

secret = "a-secret-only-bank-a-knows-at-first"
hash_lock = sha256(secret)
print(
    f"  Titus Bank's secret is private. Its hash (safe to share) = "
    f"{hash_lock[:16]}...\n"
)

supply_before_swap = (tituscoin.total_supply, novacoin.total_supply)
reserves_before_swap = (titus_issuer.reserve_usd, nova_issuer.reserve_usd)

print("=== Step 1: Titus Bank locks 1000 TTC on Chain-A (balance debited NOW) ===\n")
assert htlc_a.lock(
    sender="Titus Bank",
    receiver="Nova Bank",
    amount=1_000,
    hash_lock=hash_lock,
    timelock_height=50,
)

print(
    "\n=== Step 2: Nova Bank mirrors the lock on Chain-B "
    "-- SAME hash, SHORTER timelock ===\n"
)
assert htlc_b.lock(
    sender="Nova Bank",
    receiver="Titus Bank",
    amount=1_000,
    hash_lock=hash_lock,
    timelock_height=30,
)

assert (tituscoin.total_supply, novacoin.total_supply) == supply_before_swap
assert (titus_issuer.reserve_usd, nova_issuer.reserve_usd) == reserves_before_swap
print(
    f"\n  Mid-swap balances -- TTC: {tituscoin.balances} | "
    f"NVC: {novacoin.balances}"
)
print("  (Both balances hit zero -- funds are in escrow, belonging to neither party yet)\n")

=== SCENARIO: Titus Bank swaps 1000 TTC for Nova Bank's 1000 NVC ===

  Titus Bank's secret is private. Its hash (safe to share) = 97e635cb519f3d01...

=== Step 1: Titus Bank locks 1000 TTC on Chain-A (balance debited NOW) ===


=== Step 2: Nova Bank mirrors the lock on Chain-B -- SAME hash, SHORTER timelock ===


  Mid-swap balances -- TTC: {'Titus Bank': 0} | NVC: {'Nova Bank': 0}
  (Both balances hit zero -- funds are in escrow, belonging to neither party yet)



## 4. One public revelation unlocks both sides

> **Pause and predict:** After Titus Bank claims NVC, how can Nova Bank learn a secret that began private?

The claim record deliberately includes `revealed_secret`. Nova Bank does not receive it through a trusted messenger; it reads Chain-B's latest public block.

In [10]:
print("=== Step 3: Titus Bank claims on Chain-B, REVEALING the secret publicly ===\n")
assert htlc_b.claim(hash_lock, secret)

print(
    "\n=== Step 4: Nova Bank reads the now-public secret off Chain-B, "
    "claims on Chain-A ===\n"
)
revealed_secret = json.loads(chain_b.chain[-1].data)["revealed_secret"]
print(
    "  Nova Bank reads Chain-B's latest block, extracts "
    f"revealed_secret={revealed_secret!r}"
)
assert htlc_a.claim(hash_lock, revealed_secret)

assert tituscoin.total_supply == supply_before_swap[0]
assert novacoin.total_supply == supply_before_swap[1]
assert titus_issuer.reserve_usd == reserves_before_swap[0]
assert nova_issuer.reserve_usd == reserves_before_swap[1]
print(
    f"\n  FINAL balances -- TTC: {tituscoin.balances} | "
    f"NVC: {novacoin.balances}"
)
print(
    "  Titus Bank now holds 1000 NVC. Nova Bank now holds 1000 TTC. "
    "Fully swapped, atomically.\n"
)

for chain in (chain_a, chain_b):
    valid, message = chain.is_valid()
    print(f"  {chain.name}: valid={valid} -- {message}")

=== Step 3: Titus Bank claims on Chain-B, REVEALING the secret publicly ===

  [Chain-B] CLAIM: secret revealed on-chain = 'a-secret-only-bank-a-knows-at-first'

=== Step 4: Nova Bank reads the now-public secret off Chain-B, claims on Chain-A ===

  Nova Bank reads Chain-B's latest block, extracts revealed_secret='a-secret-only-bank-a-knows-at-first'
  [Chain-A] CLAIM: secret revealed on-chain = 'a-secret-only-bank-a-knows-at-first'

  FINAL balances -- TTC: {'Titus Bank': 0, 'Nova Bank': 1000} | NVC: {'Nova Bank': 0, 'Titus Bank': 1000}
  Titus Bank now holds 1000 NVC. Nova Bank now holds 1000 TTC. Fully swapped, atomically.

  Chain-A: valid=True -- Chain is valid.
  Chain-B: valid=True -- Chain is valid.


The swap changed **ownership only**. Neither issuer accepted a new deposit or processed a redemption, so fictional reserves stayed at 1,000 USD each and both token supplies stayed at 1,000. HTLC escrow temporarily removes balances from ordinary spending, then releases the same existing tokens; it does not mint or burn them.

## 5. Stalled swap: deadlines return both deposits

Now each issuer records another 500 fictional USD and mints 500 tokens to its own bank. Titus Bank starts another swap and then vanishes before claiming. The second lock again has the shorter deadline: height 5 on Chain-B versus height 10 on Chain-A.

In [13]:
print("\n\n=== COUNTERFACTUAL: Titus Bank locks funds, then vanishes before claiming ===\n")
print("  (Topping up balances first -- both banks spent everything in the swap above)\n")
titus_issuer.deposit_and_mint("Titus Bank", 500)
nova_issuer.deposit_and_mint("Nova Bank", 500)

secret2 = "a-swap-that-never-completes"
hash_lock2 = sha256(secret2)

assert htlc_a.lock("Titus Bank", "Nova Bank", 500, hash_lock2, timelock_height=10)
assert htlc_b.lock("Nova Bank", "Titus Bank", 500, hash_lock2, timelock_height=5)

print("\n  ... time passes. Titus Bank never claims. The secret is never revealed. ...\n")



=== COUNTERFACTUAL: Titus Bank locks funds, then vanishes before claiming ===

  (Topping up balances first -- both banks spent everything in the swap above)


  ... time passes. Titus Bank never claims. The secret is never revealed. ...



> **Model boundary:** This small manager is choreography, not a production contract. `refund` receives a caller-supplied `current_height`; it does not read a live chain clock. Meanwhile, `claim` does **not** enforce a live height at all. A real HTLC must enforce both claim and refund timing from consensus state, prevent contradictory finalization, authenticate calls, and persist contract state on-chain.

> **Pause and predict:** At modeled height 6, which refund is available? At height 11, what should the balances, reserves, and supplies be?

Chain-B's height-5 lock expires first. Chain-A's height-10 lock follows. Because no claim revealed the secret, each bank recovers its own 500 tokens.

In [16]:
print("=== Chain-B's shorter timelock expires first -- Nova Bank refunds ===\n")
assert htlc_b.refund(hash_lock2, current_height=6)

print("\n=== Chain-A's timelock expires next -- Titus Bank refunds too ===\n")
assert htlc_a.refund(hash_lock2, current_height=11)

assert tituscoin.balances == {"Titus Bank": 500, "Nova Bank": 1_000}
assert novacoin.balances == {"Nova Bank": 500, "Titus Bank": 1_000}
assert tituscoin.total_supply == 1_500
assert novacoin.total_supply == 1_500
assert titus_issuer.reserve_usd == 1_500
assert nova_issuer.reserve_usd == 1_500
assert chain_a.is_valid() == (True, "Chain is valid.")
assert chain_b.is_valid() == (True, "Chain is valid.")

print("\n  Net result: NEITHER side lost anything, even though one party did")
print("  nothing at all. That's the entire point of the timelock safety valve.")
print(f"  TTC: {tituscoin.balances} | supply={tituscoin.total_supply:.0f}")
print(f"  NVC: {novacoin.balances} | supply={novacoin.total_supply:.0f}")
print(
    f"  Fictional reserves: TTC=${titus_issuer.reserve_usd:,.0f}, "
    f"NVC=${nova_issuer.reserve_usd:,.0f}"
)

=== Chain-B's shorter timelock expires first -- Nova Bank refunds ===


=== Chain-A's timelock expires next -- Titus Bank refunds too ===


  Net result: NEITHER side lost anything, even though one party did
  nothing at all. That's the entire point of the timelock safety valve.
  TTC: {'Titus Bank': 500, 'Nova Bank': 1000} | supply=1500
  NVC: {'Nova Bank': 500, 'Titus Bank': 1000} | supply=1500
  Fictional reserves: TTC=$1,500, NVC=$1,500


## Takeaways

- **Mechanism:** a shared hash couples two independent escrows. **Not a guarantee:** that is one atomic transaction across two chains.
- **Mechanism:** revealing the secret on one chain lets the other side claim. **Not a guarantee:** the second claimer is watching, or that gossip has reached them yet.
- **Mechanism:** asymmetric deadlines give the second claimer time to react. **Not a guarantee:** the ordering was a good idea if you lock the short side first.
- **Mechanism:** refunds unwind a stall so neither side is left holding a locked bag. **Not a guarantee:** both chains halt, or a late claim races a refund — this toy does not model that fight.
- **Mechanism:** the swap moves existing tokens. **Not a guarantee:** either issuer's off-chain reserve moved. Notebook 8's vault is still invisible.

Throughout, the issuers' reserve records and each token's total supply change only during the explicit 500-unit top-ups — not during the swaps themselves.